# 02_api_shared_graph — One graph, one resource

Maxitor's six diagram endpoints all read the **same** coordinator graph — the one the load request put into the application's state. There is one per process, so it is declared once, as a single ready resource the endpoints share.

This notebook drives the real FastAPI application, without a server and without a browser.

Use the repository environment for this branch. Run these cells in order in a fresh Python kernel.

In [1]:
from __future__ import annotations

from typing import Any

from fastapi.testclient import TestClient

from aoa.action_machine.exceptions import HandleError
from aoa.maxitor.api.app import create_app
from aoa.maxitor.api.duckdb_graph_holder import DuckDBGraphHolder
from aoa.maxitor.model.diagrams.resources.duckdb_graph_resource import DuckDBGraphResource

## The fixture

One domain and one two-state lifecycle — small enough to read, and enough for every endpoint.

In [2]:
def _payload(domain_id: str) -> dict[str, Any]:
    """Return one domain and one two-state lifecycle."""
    lifecycle = "e.x.Entity:lifecycle:status"
    return {
        "schema_version": "1.0",
        "nodes": [
            {"id": domain_id, "label": domain_id, "type": "Domain",
             "properties": {"name": domain_id, "description": ""}},
            {"id": lifecycle, "label": "status", "type": "Lifecycle",
             "properties": {"field_name": "status"}},
            {"id": f"{lifecycle}:draft", "label": "draft", "type": "StateInitial",
             "properties": {"lifecycle_class_id": "x.Status", "state_key": "draft"}},
            {"id": f"{lifecycle}:done", "label": "done", "type": "StateFinal",
             "properties": {"lifecycle_class_id": "x.Status", "state_key": "done"}},
        ],
        "edges": [
            {"source_id": lifecycle, "target_id": f"{lifecycle}:draft",
             "type": "lifecycle_contains_state", "relationship": "COMPOSITION",
             "is_dag": False, "properties": {"state_key": "draft"}},
            {"source_id": lifecycle, "target_id": f"{lifecycle}:done",
             "type": "lifecycle_contains_state", "relationship": "COMPOSITION",
             "is_dag": False, "properties": {"state_key": "done"}},
            {"source_id": f"{lifecycle}:draft", "target_id": f"{lifecycle}:done",
             "type": "lifecycle_transition", "relationship": "COMPOSITION",
             "is_dag": False, "properties": {"from_state": "draft", "to_state": "done"}},
        ],
    }

## 1. Nothing loaded yet

A caller reaches an endpoint before anything is loaded. The adapter answers every uncaught exception with one generic body — but the failure behind it is the one this feature changed.

In [3]:
app = create_app()
client = TestClient(app, raise_server_exceptions=False)

answer = client.get("/api/v1/list-domains")
print(f"GET /api/v1/list-domains -> {answer.status_code} {answer.text}")

GET /api/v1/list-domains -> 500 {"detail":"Internal server error"}


Driving the resource directly — as a developer or a test would — shows the sentence the caller never sees:

In [4]:
try:
    _ = DuckDBGraphHolder(app).service
except HandleError as exc:
    print(f"{type(exc).__name__}: {exc}")

HandleError: Graph is not loaded: call POST /api/load first.


## 2. Load a graph

In real use this is `POST /api/load`, which fetches the coordinator model from a service. Here the graph is built from the fixture.

In [5]:
first = DuckDBGraphResource.build_from_json(_payload("d.shop"))
app.state.duckdb_graph = first
print(first.execute_fetch_dicts("SELECT id FROM domain"))

[{'id': 'd.shop'}]


## 3. All six endpoints answer, from one shared resource

In [6]:
DIAGRAM_ROUTES = (
    ("/api/v1/list-domains", {}),
    ("/api/v1/list-entities", {}),
    ("/api/v1/list-node-types", {}),
    ("/api/v1/full-graph", {}),
    ("/api/v1/lifecycle-finite-automaton", {"lifecycle_graph_node_id": "e.x.Entity:lifecycle:status"}),
    ("/api/v1/domain-use-case-diagram", {"domain_id": "d.shop"}),
)

for path, query in DIAGRAM_ROUTES:
    print(f"{path:38s} -> {client.get(path, params=query).status_code}")

/api/v1/list-domains                   -> 200
/api/v1/list-entities                  -> 200
/api/v1/list-node-types                -> 200
/api/v1/full-graph                     -> 200
/api/v1/lifecycle-finite-automaton     -> 200
/api/v1/domain-use-case-diagram        -> 200


A counting stand-in proves they read that one object, and that reading is what resolves it:

In [7]:
class _CountingGraph:
    """A graph stand-in that counts how often an endpoint reached it."""

    def __init__(self, inner: DuckDBGraphResource) -> None:
        self.inner = inner
        self.reads = 0

    @property
    def service(self) -> Any:
        self.reads += 1
        return self.inner.service

    def execute_fetch_dicts(self, sql: str, parameters: Any = None) -> Any:
        self.reads += 1
        return self.inner.execute_fetch_dicts(sql, parameters)


counting = _CountingGraph(DuckDBGraphResource.build_from_json(_payload("d.shop")))
app.state.duckdb_graph = counting
for path, query in DIAGRAM_ROUTES:
    client.get(path, params=query)
print(f"the six requests together read the loaded graph {counting.reads} times")

the six requests together read the loaded graph 14 times


## 4. A second load replaces it for everyone

The shared resource resolves on every read and caches nothing, so nothing is restarted and no endpoint holds a graph of its own.

In [8]:
second = DuckDBGraphResource.build_from_json(_payload("d.warehouse"))
app.state.duckdb_graph = second
after = client.get("/api/v1/list-domains")
print(f"GET /api/v1/list-domains -> {after.status_code} {after.text}")

GET /api/v1/list-domains -> 200 {"list_domains":[{"qualname":"d.warehouse","label":"d.warehouse","color":"#3b82f6"}]}
